# Clean BFA/BFI Split Dashboard Generator

This notebook generates a clean interactive HTML dashboard from:

1. a full `.mp4` video;
2. an activity CSV with start/end times;
3. a folder or zip containing split BFA `.npy/.npz` files and their matching timestamp files.

## How to use

1. Fill only the fields in the **User inputs** cell below.
2. Run all cells.
3. The generated HTML will be downloaded automatically at the end.

## Input modes

- `drive`: use files already available in Google Drive. In this mode, `SPLIT_NPY_DIR` must be a **folder** containing all split BFA files and their matching timestamp files.
- `upload`: upload the video, CSV, and either individual `.npy/.npz` files or a `.zip` containing them.

## What this notebook assumes

- The dashboard always uses **BFA** files: `*_bfa.npy` or `*_bfa.npz`.
- Timestamp files are expected in the same folder as the BFA files.
- The video is embedded in the HTML.
- The PCAP is not required.

In [76]:
#@title User inputs

INPUT_MODE = "drive" #@param ["drive", "upload"]
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}

VIDEO_PATH = "/content/drive/MyDrive/WIFI/BeamHAR-SVR-Data/scenarios/lab/p03/s01/raw/video_cam01.mp4" #@param {type:"string"}
ACTIVITY_CSV_PATH = "/content/drive/MyDrive/WIFI/BeamHAR-SVR-Data/scenarios/lab/p03/s01/annotations/activity_segments.csv" #@param {type:"string"}

# In drive mode, this must be a folder containing all split *_bfa.npy files
# and their matching *_timestamp.npy files.
# In upload mode, you may upload either individual .npy/.npz files or one .zip.
SPLIT_NPY_DIR = "/content/drive/MyDrive/WIFI/BeamHAR-SVR-Data/scenarios/lab/p03/s01/processed" #@param {type:"string"}

# Choose one station, for example: "01", "02", or "03".
TARGET_STATION = "02" #@param {type:"string"}

## 1. Setup

In [77]:
import os
import re
import json
import base64
import zipfile
import mimetypes
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

try:
    from google.colab import files, drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PREFERRED_SPLIT_TYPE = "BFA"
EMBED_VIDEO = True
MAX_EMBED_VIDEO_MB = 300
SHOW_MAIN_HEATMAP = True
SHOW_ACTIVITY_HEATMAPS = True
SHOW_MAPPING_DETAILS = False

WINDOW_SEC = 3.0
MAX_FEATURES_HTML = 300
MAX_TIME_POINTS_HTML = 2500
MAX_FEATURES_ACTIVITY = 600
MAX_TIME_ACTIVITY = 1600

USE_MP4_CREATION_TIME_TO_ALIGN_CSV = True
MP4_CREATION_TIME_MEANS_VIDEO_END = True
CSV_TIMEZONE = "America/Sao_Paulo"
MANUAL_VIDEO_START_DATETIME = ""
MANUAL_CSV_TIME_OFFSET_SEC = None

MAPPING_TIME_TOLERANCE_SEC = 0.20
SKIP_ROWS_WITH_NO_VALID_SPLIT = True

NORMALIZATION = "feature_minmax"
COMPLEX_MODE = "abs"

OUTPUT_DIR = Path("/content/bfa_bfi_dashboard_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_HTML = OUTPUT_DIR / "bfa_bfi_dashboard.html"
OUTPUT_ZIP = OUTPUT_DIR / "bfa_bfi_dashboard.zip"

INPUT_DIR = Path("/content/bfa_bfi_dashboard_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR = INPUT_DIR / "extracted"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", IN_COLAB)

Running in Colab: True


## 2. Load input files

In [78]:
def extract_zip(zip_path, output_dir):
    target = Path(output_dir) / Path(zip_path).stem
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(target)
    return target

if INPUT_MODE == "drive":
    if IN_COLAB and MOUNT_GOOGLE_DRIVE:
        drive.mount("/content/drive")

    VIDEO_PATH = Path(VIDEO_PATH)
    ACTIVITY_CSV_PATH = Path(ACTIVITY_CSV_PATH)
    SPLIT_ARRAY_DIR = Path(SPLIT_NPY_DIR)
    SPLIT_TIMESTAMP_DIR = SPLIT_ARRAY_DIR

    if not VIDEO_PATH.exists():
        raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
    if not ACTIVITY_CSV_PATH.exists():
        raise FileNotFoundError(f"Activity CSV not found: {ACTIVITY_CSV_PATH}")
    if not SPLIT_ARRAY_DIR.exists() or not SPLIT_ARRAY_DIR.is_dir():
        raise ValueError("In drive mode, SPLIT_NPY_DIR must be a folder containing all split .npy/.npz files.")

elif INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Upload mode only works in Google Colab.")

    uploaded = files.upload()
    for name, data in uploaded.items():
        (INPUT_DIR / name).write_bytes(data)

    videos = sorted(INPUT_DIR.glob("*.mp4"))
    csvs = sorted(INPUT_DIR.glob("*.csv"))
    zips = sorted(INPUT_DIR.glob("*.zip"))

    if not videos:
        raise FileNotFoundError("No .mp4 video was uploaded.")
    if not csvs:
        raise FileNotFoundError("No activity CSV was uploaded.")

    VIDEO_PATH = videos[0]
    ACTIVITY_CSV_PATH = csvs[0]

    if zips:
        for zp in zips:
            extract_zip(zp, EXTRACT_DIR)
        SPLIT_ARRAY_DIR = EXTRACT_DIR
        SPLIT_TIMESTAMP_DIR = EXTRACT_DIR
    else:
        SPLIT_ARRAY_DIR = INPUT_DIR
        SPLIT_TIMESTAMP_DIR = INPUT_DIR
else:
    raise ValueError("INPUT_MODE must be 'drive' or 'upload'.")

print("Video:", VIDEO_PATH)
print("Activity CSV:", ACTIVITY_CSV_PATH)
print("Split array directory:", SPLIT_ARRAY_DIR)
print("Timestamp directory: same as split array directory")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Video: /content/drive/MyDrive/WIFI/BeamHAR-SVR-Data/scenarios/lab/p03/s01/raw/video_cam01.mp4
Activity CSV: /content/drive/MyDrive/WIFI/BeamHAR-SVR-Data/scenarios/lab/p03/s01/annotations/activity_segments.csv
Split array directory: /content/drive/MyDrive/WIFI/BeamHAR-SVR-Data/scenarios/lab/p03/s01/processed
Timestamp directory: same as split array directory


## 3. Read and align the activity CSV

In [79]:
def normalize_text(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())

def slugify(value):
    value = normalize_text(value)
    replacements = {"ç":"c","ã":"a","á":"a","à":"a","â":"a","é":"e","ê":"e","í":"i","ó":"o","ô":"o","õ":"o","ú":"u"}
    for src, dst in replacements.items():
        value = value.replace(src, dst)
    value = value.replace("/", "_").replace("-", "_")
    value = re.sub(r"[^a-z0-9_]+", "_", value)
    return re.sub(r"_+", "_", value).strip("_")

def find_column(df, candidates):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for item in candidates:
        if item.lower() in lower_map:
            return lower_map[item.lower()]
    for col in df.columns:
        if any(item.lower() in col.lower() for item in candidates):
            return col
    return None

def parse_time_column(series):
    raw = series.astype(str).str.strip()
    numeric = pd.to_numeric(raw.str.replace(",", ".", regex=False), errors="coerce")
    if numeric.notna().mean() > 0.8:
        return numeric.astype(float), "seconds"
    cleaned = raw.str.replace(r"(\d{2}:\d{2}:\d{2}),(\d+)", r"\1.\2", regex=True)
    datetimes = pd.to_datetime(cleaned, errors="coerce")
    if datetimes.notna().mean() > 0.8:
        return datetimes, "datetime"
    raise ValueError("Could not parse time column.")

def ffprobe_value(args):
    try:
        result = subprocess.run(args, check=True, capture_output=True, text=True)
        return result.stdout.strip()
    except Exception:
        return ""

def video_duration_seconds(path):
    value = ffprobe_value(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)])
    return float(value) if value else None

def video_creation_time(path):
    value = ffprobe_value(["ffprobe", "-v", "error", "-show_entries", "format_tags=creation_time", "-of", "default=noprint_wrappers=1:nokey=1", str(path)])
    return pd.to_datetime(value.splitlines()[0], utc=True) if value else None

activity_raw = pd.read_csv(ACTIVITY_CSV_PATH)

start_col = find_column(activity_raw, ["start_time", "start", "start_sec", "start_s", "begin"])
end_col = find_column(activity_raw, ["end_time", "end", "end_sec", "end_s", "finish"])
label_col = find_column(activity_raw, ["label", "activity", "class", "description", "annotation"])
person_col = find_column(activity_raw, ["person", "subject", "participant"])

if start_col is None or end_col is None or label_col is None:
    raise ValueError(f"The CSV must contain start, end, and label columns. Found: {list(activity_raw.columns)}")

start_values, start_kind = parse_time_column(activity_raw[start_col])
end_values, end_kind = parse_time_column(activity_raw[end_col])

if start_kind == "datetime" and end_kind == "datetime":
    if MANUAL_VIDEO_START_DATETIME:
        reference_time = pd.to_datetime(MANUAL_VIDEO_START_DATETIME)
        alignment_method = "manual_video_start_datetime"
    elif USE_MP4_CREATION_TIME_TO_ALIGN_CSV:
        creation_utc = video_creation_time(VIDEO_PATH)
        duration_sec = video_duration_seconds(VIDEO_PATH)
        if creation_utc is not None and duration_sec is not None:
            creation_local = creation_utc.tz_convert(CSV_TIMEZONE).tz_localize(None)
            reference_time = creation_local - pd.to_timedelta(duration_sec, unit="s") if MP4_CREATION_TIME_MEANS_VIDEO_END else creation_local
            alignment_method = "mp4_creation_time_minus_duration" if MP4_CREATION_TIME_MEANS_VIDEO_END else "mp4_creation_time_as_start"
        else:
            reference_time = start_values.min()
            alignment_method = "fallback_first_csv_start"
    else:
        reference_time = start_values.min()
        alignment_method = "first_csv_start"

    start_seconds = (start_values - reference_time).dt.total_seconds()
    end_seconds = (end_values - reference_time).dt.total_seconds()
else:
    reference_time = None
    alignment_method = "numeric_seconds"
    start_seconds = start_values.astype(float)
    end_seconds = end_values.astype(float)

if MANUAL_CSV_TIME_OFFSET_SEC is not None:
    start_seconds += float(MANUAL_CSV_TIME_OFFSET_SEC)
    end_seconds += float(MANUAL_CSV_TIME_OFFSET_SEC)

group_map = {
    "sitting": "sitting",
    "sitting down": "sitting_down",
    "sentado": "sitting",
    "walking": "walking",
    "andando": "walking",
    "lying down": "lying_down",
    "deitado": "lying_down",
    "lying down falling": "lying_down_falling",
    "lying down/falling": "lying_down_falling",
    "falling": "lying_down_falling",
    "standing up": "standing_up",
    "standing up from chair": "standing_up",
    "standing up from floor": "standing_up",
}

def label_to_group(label):
    normalized = normalize_text(label).replace("/", " ")
    return group_map.get(normalized, slugify(label))

activity_df = pd.DataFrame({
    "start": start_seconds.astype(float),
    "end": end_seconds.astype(float),
    "label": activity_raw[label_col].astype(str),
    "start_datetime": start_values if start_kind == "datetime" else pd.NaT,
})
activity_df["group"] = activity_df["label"].map(label_to_group)
activity_df["label_slug"] = activity_df["label"].map(slugify)
activity_df["person"] = activity_raw[person_col].astype(str) if person_col else ""
activity_df = activity_df.sort_values("start").reset_index(drop=True)

print("CSV/video alignment:", alignment_method)
print("Video start reference:", reference_time)
print("First CSV event in video seconds:", float(activity_df.iloc[0]["start"]))
display(activity_df.head())

CSV/video alignment: mp4_creation_time_minus_duration
Video start reference: 2026-05-11 14:29:45.993000
First CSV event in video seconds: 0.56


,start,end,label,start_datetime,group,label_slug,person
0,0.560,1.493,standing up (from chair),2026-05-11 14:29:46.553,standing_up_from_chair,standing_up_from_chair,p03
1,1.526,1.493,sitting down,2026-05-11 14:29:47.519,sitting_down,sitting_down,p03
2,1.526,5.860,walking,2026-05-11 14:29:47.519,walking,walking,p03
3,5.893,7.860,lying down/falling,2026-05-11 14:29:51.886,lying_down_falling,lying_down_falling,p03
4,7.893,9.293,lying down,2026-05-11 14:29:53.886,lying_down,lying_down,p03


## 4. Map CSV rows to split arrays

In [80]:
def is_timestamp_file(path):
    return Path(path).name.lower().endswith(("_timestamp.npy", "_timestamps.npy", "_timestamp.npz", "_timestamps.npz"))

def is_bfa_file(path):
    return Path(path).name.lower().endswith(("_bfa.npy", "_bfa.npz"))

def is_bfi_v_file(path):
    return Path(path).name.lower().endswith(("_v.npy", "_v.npz"))

def parse_split_filename(path):
    stem = Path(path).stem
    match = re.match(
        r"(?P<label>.+?)_(?P<person>p\d+)_sta(?P<station>\d+)_(?P<dt>\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2},\d+)_(?P<kind>bfa|v|timestamp|timestamps)$",
        stem,
        flags=re.IGNORECASE,
    )
    if not match:
        return {"parsed": False, "file_label": "", "file_label_slug": "", "station": "", "file_datetime": pd.NaT, "kind": ""}
    dt = pd.to_datetime(match.group("dt").replace(",", "."), format="%Y-%m-%d-%H-%M-%S.%f", errors="coerce")
    return {
        "parsed": True,
        "file_label": match.group("label"),
        "file_label_slug": slugify(match.group("label").replace("/", " ")),
        "station": match.group("station"),
        "file_datetime": dt,
        "kind": match.group("kind").lower(),
    }

def load_array(path):
    path = Path(path)
    if path.suffix.lower() == ".npz":
        loaded = np.load(path, allow_pickle=False)
        key = sorted(loaded.files, key=lambda k: loaded[k].size, reverse=True)[0]
        return np.asarray(loaded[key])
    return np.asarray(np.load(path, allow_pickle=False))

def array_shape(path):
    try:
        if Path(path).suffix.lower() == ".npz":
            loaded = np.load(path, allow_pickle=False)
            key = sorted(loaded.files, key=lambda k: loaded[k].size, reverse=True)[0]
            return tuple(loaded[key].shape)
        return tuple(np.load(path, mmap_mode="r", allow_pickle=False).shape)
    except Exception as exc:
        return f"ERROR: {exc}"

def valid_data_array(path):
    shape = array_shape(path)
    return isinstance(shape, tuple) and len(shape) >= 2 and shape[0] > 0

def label_matches(csv_label, file_label):
    return slugify(str(csv_label).replace("/", " ")) == slugify(str(file_label).replace("/", " "))

all_arrays = sorted(list(Path(SPLIT_ARRAY_DIR).rglob("*.npy")) + list(Path(SPLIT_ARRAY_DIR).rglob("*.npz")))
timestamp_files = [p for p in all_arrays if is_timestamp_file(p)]

if PREFERRED_SPLIT_TYPE == "BFA":
    data_candidates = [p for p in all_arrays if is_bfa_file(p) and valid_data_array(p)]
else:
    data_candidates = [p for p in all_arrays if is_bfi_v_file(p) and valid_data_array(p)]

if TARGET_STATION:
    data_candidates = [p for p in data_candidates if parse_split_filename(p)["station"] == str(TARGET_STATION).zfill(2)]

if not data_candidates:
    raise FileNotFoundError("No valid split arrays were found after filtering.")

def matching_timestamp_file(data_file):
    name = Path(data_file).name
    if is_bfa_file(data_file):
        target = name.replace("_bfa.npy", "_timestamp.npy").replace("_bfa.npz", "_timestamp.npz")
    else:
        target = name.replace("_v.npy", "_timestamp.npy").replace("_v.npz", "_timestamp.npz")
    matches = [p for p in timestamp_files if p.name == target]
    return str(matches[0]) if matches else ""

mapping_rows = []
used_files = set()

for index, row in activity_df.iterrows():
    candidates = [p for p in data_candidates if p not in used_files]
    scored = []

    for path in candidates:
        info = parse_split_filename(path)
        if not info["parsed"] or not label_matches(row["label"], info["file_label"]):
            continue
        if pd.isna(row["start_datetime"]) or pd.isna(info["file_datetime"]):
            time_diff = 0.0
        else:
            time_diff = abs((info["file_datetime"] - row["start_datetime"]).total_seconds())
        scored.append((time_diff, path, info))

    scored.sort(key=lambda item: item[0])

    if not scored:
        if SKIP_ROWS_WITH_NO_VALID_SPLIT:
            print(f"Skipping CSV row {index}: no matching valid split.")
            continue
        raise FileNotFoundError(f"No matching split for CSV row {index}: {row['label']}")

    best_diff, data_file, info = scored[0]

    if best_diff > MAPPING_TIME_TOLERANCE_SEC:
        message = f"Skipping CSV row {index}: closest valid split is {best_diff:.3f}s away."
        if SKIP_ROWS_WITH_NO_VALID_SPLIT:
            print(message)
            continue
        raise ValueError(message)

    used_files.add(data_file)
    mapping_rows.append({
        "csv_index": index,
        "start": row["start"],
        "end": row["end"],
        "label": row["label"],
        "group": row["group"],
        "data_file": str(data_file),
        "data_shape": str(array_shape(data_file)),
        "file_datetime": info["file_datetime"],
        "time_difference_sec": best_diff,
        "timestamp_file": matching_timestamp_file(data_file),
    })

mapping_df = pd.DataFrame(mapping_rows)

if mapping_df.empty:
    raise ValueError("No valid CSV row was mapped to a split array.")

display(mapping_df[["csv_index", "label", "group", "data_shape", "time_difference_sec"]].head())
print("Mapped splits:", len(mapping_df))
print("Skipped CSV rows:", len(activity_df) - len(mapping_df))

Skipping CSV row 1: closest valid split is 15.200s away.
Skipping CSV row 11: closest valid split is 40.266s away.
Skipping CSV row 12: closest valid split is 40.233s away.
Skipping CSV row 15: closest valid split is 100.633s away.
Skipping CSV row 16: closest valid split is 62.400s away.
Skipping CSV row 17: closest valid split is 62.966s away.
Skipping CSV row 18: closest valid split is 21.500s away.
Skipping CSV row 19: closest valid split is 20.866s away.
Skipping CSV row 20: closest valid split is 20.666s away.
Skipping CSV row 21: closest valid split is 20.333s away.
Skipping CSV row 22: closest valid split is 10.833s away.
Skipping CSV row 23: closest valid split is 79.933s away.
Skipping CSV row 24: closest valid split is 41.133s away.
Skipping CSV row 25: closest valid split is 41.633s away.
Skipping CSV row 31: closest valid split is 59.167s away.
Skipping CSV row 32: closest valid split is 21.000s away.
Skipping CSV row 33: closest valid split is 21.566s away.
Skipping CSV r

,csv_index,label,group,data_shape,time_difference_sec
0,0,standing up (from chair),standing_up_from_chair,"(5, 234, 6)",0.0
1,2,walking,walking,"(77, 234, 6)",0.0
2,3,lying down/falling,lying_down_falling,"(35, 234, 6)",0.0
3,4,lying down,lying_down,"(27, 234, 6)",0.0
4,5,standing up (from floor),standing_up_from_floor,"(49, 234, 6)",0.0


Mapped splits: 36
Skipped CSV rows: 63


## 5. Join arrays and compute metrics

In [81]:
def array_to_features(array):
    array = np.asarray(array)
    if array.ndim < 2:
        raise ValueError("Each split array must have at least two dimensions.")
    if np.iscomplexobj(array):
        values = np.abs(array) if COMPLEX_MODE == "abs" else np.concatenate([array.real, array.imag], axis=-1)
    else:
        values = array.astype(float)
    return values.reshape(values.shape[0], -1)

def split_time_axis(num_samples, start, end, timestamp_file):
    if timestamp_file and str(timestamp_file).strip():
        timestamps = np.atleast_1d(np.asarray(load_array(timestamp_file), dtype=float).squeeze())
        if timestamps.size == num_samples:
            if timestamps.size > 0 and np.nanmax(timestamps) > 1e6:
                timestamps = timestamps - timestamps[0] + float(start)
            return timestamps, "timestamp_file"
    return np.linspace(float(start), float(end), num_samples), "interpolated_from_csv"

feature_parts = []
time_parts = []
metadata_rows = []
raw_arrays = []
time_sources = []
expected_features = None

for _, row in mapping_df.iterrows():
    array = load_array(row["data_file"])
    features = array_to_features(array)

    if expected_features is None:
        expected_features = features.shape[1]
    elif features.shape[1] != expected_features:
        raise ValueError(f"Incompatible feature count in {row['data_file']}.")

    times, source = split_time_axis(features.shape[0], row["start"], row["end"], row["timestamp_file"])

    feature_parts.append(features)
    time_parts.append(times)
    raw_arrays.append(array)
    time_sources.append(source)
    metadata_rows.extend([{"label": row["label"], "group": row["group"], "start": row["start"], "end": row["end"], "data_file": Path(row["data_file"]).name, "time_source": source}] * features.shape[0])

feature_matrix_raw = np.vstack(feature_parts)
time_axis = np.concatenate(time_parts)
sample_metadata_df = pd.DataFrame(metadata_rows)

order = np.argsort(time_axis)
feature_matrix_raw = feature_matrix_raw[order]
time_axis = time_axis[order]
sample_metadata_df = sample_metadata_df.iloc[order].reset_index(drop=True)

def normalize_matrix(matrix):
    matrix = np.asarray(matrix, dtype=float)
    if NORMALIZATION == "none":
        return matrix
    if NORMALIZATION == "global_minmax":
        mn, mx = np.nanmin(matrix), np.nanmax(matrix)
        return (matrix - mn) / (mx - mn) if mx > mn else np.zeros_like(matrix)
    mn = np.nanmin(matrix, axis=0, keepdims=True)
    mx = np.nanmax(matrix, axis=0, keepdims=True)
    den = mx - mn
    den[~np.isfinite(den) | (den == 0)] = 1
    return np.nan_to_num((matrix - mn) / den)

feature_matrix = normalize_matrix(feature_matrix_raw)
num_samples, num_features = feature_matrix.shape

rms_series = np.sqrt(np.mean(np.square(feature_matrix), axis=1))
delta_series = np.zeros(num_samples)
if num_samples > 1:
    delta_series[1:] = np.sqrt(np.mean(np.square(np.diff(feature_matrix, axis=0)), axis=1))

def angle_names(array):
    if np.asarray(array).ndim < 3:
        return []
    n = np.asarray(array).shape[-1]
    if n == 6:
        return ["phi_11", "phi_21", "phi_31", "psi_21", "psi_31", "psi_41"]
    if n == 4:
        return ["phi_11", "phi_21", "psi_21", "psi_31"]
    return [f"component_{i}" for i in range(n)]

ANGLE_NAMES = angle_names(raw_arrays[0])

def angle_variation(raw_arrays):
    if not ANGLE_NAMES:
        return None
    chunks = []
    for array in raw_arrays:
        values = np.abs(array) if np.iscomplexobj(array) else array.astype(float)
        if values.ndim < 3:
            return None
        diff = np.diff(values, axis=0)
        axes = tuple(range(1, diff.ndim - 1))
        d = np.sqrt(np.mean(np.square(diff), axis=axes)) if axes else np.abs(diff)
        d = np.vstack([np.zeros((1, d.shape[-1])), d])
        chunks.append(d[:, :len(ANGLE_NAMES)])
    result = np.vstack(chunks)[order]
    for j in range(result.shape[1]):
        mn, mx = np.nanmin(result[:, j]), np.nanmax(result[:, j])
        result[:, j] = (result[:, j] - mn) / (mx - mn) if mx > mn else 0
    return result

angle_delta_norm = angle_variation(raw_arrays)

summary_df = pd.DataFrame([{
    "data_type": PREFERRED_SPLIT_TYPE,
    "station": TARGET_STATION,
    "mapped_splits": len(mapping_df),
    "samples": num_samples,
    "features": num_features,
    "matrix_shape": str(feature_matrix.shape),
    "time_source": " + ".join(sorted(set(time_sources))),
    "normalization": NORMALIZATION,
    "angles_or_components": ", ".join(ANGLE_NAMES) if ANGLE_NAMES else "not applicable",
}])

display(summary_df)

,data_type,station,mapped_splits,samples,features,matrix_shape,time_source,normalization,angles_or_components
0,BFA,02,36,1206,1404,"(1206, 1404)",timestamp_file,feature_minmax,"phi_11, phi_21, phi_31, psi_21, psi_31, psi_41"


## 6. Generate activity heatmaps

In [82]:
STATIC_IMAGE_DIR = OUTPUT_DIR / "activity_heatmaps"
STATIC_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def downsample_indices(length, max_length):
    if length <= max_length:
        return np.arange(length)
    return np.unique(np.linspace(0, length - 1, max_length).astype(int))

activity_heatmaps = []
group_order = list(dict.fromkeys(sample_metadata_df["group"].tolist()))

for group in group_order:
    idx = np.where(sample_metadata_df["group"].values == group)[0]
    if len(idx) < 2:
        continue

    matrix = feature_matrix[idx]
    times = time_axis[idx]
    ti = downsample_indices(matrix.shape[0], MAX_TIME_ACTIVITY)
    fi = downsample_indices(matrix.shape[1], MAX_FEATURES_ACTIVITY)

    output_png = STATIC_IMAGE_DIR / f"activity_heatmap_{group}.png"

    plt.figure(figsize=(12, 5))
    plt.imshow(
        matrix[np.ix_(ti, fi)].T,
        aspect="auto",
        origin="lower",
        extent=[float(times[ti].min()), float(times[ti].max()), int(fi.min()), int(fi.max())],
    )
    plt.colorbar(label="Normalized intensity")
    plt.xlabel("Video time (s)")
    plt.ylabel("BFA/BFI features")
    plt.title(f"Activity heatmap: {group}")
    plt.tight_layout()
    plt.savefig(output_png, dpi=180, bbox_inches="tight")
    plt.close()

    activity_heatmaps.append({
        "group": group,
        "file": output_png,
        "sample_count": len(idx),
        "description": (
            "This heatmap shows the BFA/BFI feature pattern for this activity. "
            "The x-axis is video time, the y-axis is the selected feature index, and the color is the normalized feature value. "
            "Use it to compare how the feature pattern changes across activities."
        ),
    })

print("Activity heatmaps generated:", len(activity_heatmaps))

Activity heatmaps generated: 7


## 7. Generate HTML

In [83]:
def file_to_data_uri(path):
    path = Path(path)
    mime = mimetypes.guess_type(str(path))[0] or "application/octet-stream"
    return f"data:{mime};base64," + base64.b64encode(path.read_bytes()).decode("ascii")

def video_block(path):
    path = Path(path)
    size_mb = path.stat().st_size / (1024 * 1024)
    if EMBED_VIDEO and size_mb <= MAX_EMBED_VIDEO_MB:
        return f'<video id="video" controls style="width:100%;max-height:42vh;background:#000;"><source src="{file_to_data_uri(path)}"></video>', f"Embedded video: {path.name} ({size_mb:.1f} MB)"
    return '<div class="note">Video was not embedded. Use the slider.</div>', f"Video not embedded: {path.name} ({size_mb:.1f} MB)."

def safe_array(array):
    return np.nan_to_num(np.asarray(array, dtype=float), nan=0.0, posinf=0.0, neginf=0.0).tolist()

def image_gallery_records(items):
    return [{"group": item["group"], "src": file_to_data_uri(item["file"]), "sample_count": int(item["sample_count"]), "description": item["description"]} for item in items]

t_idx = downsample_indices(num_samples, MAX_TIME_POINTS_HTML)
f_idx = downsample_indices(num_features, MAX_FEATURES_HTML)

payload = {
    "x": safe_array(time_axis[t_idx]),
    "y": [int(v) for v in f_idx],
    "z": safe_array(feature_matrix[np.ix_(t_idx, f_idx)].T),
    "rms": safe_array(rms_series[t_idx]),
    "delta": safe_array(delta_series[t_idx]),
    "cumTimes": safe_array(time_axis),
    "cumCount": safe_array(np.arange(1, num_samples + 1)),
    "cumFraction": safe_array(np.arange(1, num_samples + 1) / num_samples),
    "angleNames": ANGLE_NAMES,
    "angleDelta": safe_array(angle_delta_norm[t_idx, :len(ANGLE_NAMES)].T) if angle_delta_norm is not None else [],
    "hasAnglePlot": angle_delta_norm is not None,
    "activities": [{"start": float(r.start), "end": float(r.end), "label": str(r.label), "group": str(r.group)} for _, r in activity_df.iterrows()],
    "showMainHeatmap": bool(SHOW_MAIN_HEATMAP),
    "tmin": float(np.min(time_axis[t_idx])),
    "tmax": float(np.max(time_axis[t_idx])),
    "windowSec": float(WINDOW_SEC),
}

video_html, video_status = video_block(VIDEO_PATH)

inputs_html = pd.DataFrame([
    {"input": "video", "value": str(VIDEO_PATH), "note": video_status},
    {"input": "activity_csv", "value": str(ACTIVITY_CSV_PATH), "note": f"{len(activity_df)} rows"},
    {"input": "split_folder", "value": str(SPLIT_ARRAY_DIR), "note": f"{len(mapping_df)} mapped splits; timestamps searched in same folder"},
    {"input": "data_type", "value": PREFERRED_SPLIT_TYPE, "note": f"station {TARGET_STATION}"},
]).to_html(index=False, escape=True, classes="data-table")

mapping_html = mapping_df.to_html(index=False, escape=True, classes="data-table")
summary_html = summary_df.to_html(index=False, escape=True, classes="data-table")

matrix_example = pd.DataFrame(feature_matrix[:5, :10], columns=[f"feature_{i}" for i in range(min(10, num_features))])
matrix_example.insert(0, "sample", np.arange(len(matrix_example)))
matrix_example_html = matrix_example.to_html(index=False, escape=True, classes="data-table")

first_array = raw_arrays[0]
if np.asarray(first_array).ndim >= 3:
    vector = np.asarray(first_array[0, 0, :]).reshape(-1)
    names = ANGLE_NAMES if len(ANGLE_NAMES) == len(vector) else [f"component_{i}" for i in range(len(vector))]
    split_example_html = pd.DataFrame({"position": range(len(vector)), "angle_or_component": names, "value": [str(v) for v in vector]}).to_html(index=False, escape=True, classes="data-table")
else:
    split_example_html = "<p>The first split does not have an explicit angle/component dimension.</p>"

activity_gallery_html = ""
for rec in image_gallery_records(activity_heatmaps if SHOW_ACTIVITY_HEATMAPS else []):
    activity_gallery_html += f"""
    <div class="gallery-card">
      <img src="{rec['src']}">
      <div class="caption">{rec['group']}</div>
      <p class="small"><b>Samples:</b> {rec['sample_count']}</p>
      <p class="small">{rec['description']}</p>
    </div>
    """
if not activity_gallery_html:
    activity_gallery_html = "<p>No activity heatmap was generated.</p>"

mapping_open = ""
mapping_section = f"""
<details {mapping_open}>
  <summary>CSV-to-split mapping (diagnostic, click to expand)</summary>
  <p class="small">This diagnostic table shows which split array was matched to each CSV row. It is collapsed by default to avoid taking too much space.</p>
  {mapping_html}
</details>
"""

html_template = r"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Clean BFA/BFI Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body { font-family: Arial, sans-serif; margin:0; background:#f6f7f9; color:#111; }
.header { background:#111; color:white; padding:12px 18px; display:flex; gap:18px; flex-wrap:wrap; align-items:center; }
.header h2 { margin:0; font-size:20px; }
.meta { font-size:13px; opacity:.9; }
.dashboard { width:98.5vw; margin:12px auto; display:grid; grid-template-columns:36% 64%; gap:12px; }
.card { background:white; border:1px solid #ddd; border-radius:12px; padding:12px; box-shadow:0 1px 4px rgba(0,0,0,.08); }
.video-card { position:sticky; top:8px; }
.badge { display:inline-block; padding:7px 9px; background:#f5f5f5; border-radius:8px; margin:4px 4px 0 0; }
.grid2 { display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-top:12px; }
.small { font-size:13px; color:#444; }
.note { padding:8px 10px; background:#eaf4ff; border:1px solid #b8d8ff; border-radius:8px; }
.gallery { width:98.5vw; margin:12px auto; display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:16px; }
.gallery-card { background:white; border:1px solid #ddd; border-radius:12px; padding:12px; }
.gallery-card img { width:100%; border:1px solid #ddd; border-radius:8px; }
.caption { font-weight:bold; margin-top:8px; }
.tables { width:98.5vw; margin:12px auto 28px auto; }
.data-table { border-collapse:collapse; width:100%; font-size:13px; margin:10px 0 18px 0; }
.data-table th, .data-table td { border:1px solid #ddd; padding:6px 8px; vertical-align:top; }
.data-table th { background:#f1f1f1; text-align:left; }
summary { cursor:pointer; font-weight:bold; margin:8px 0; }
@media(max-width:1200px) { .dashboard { grid-template-columns:1fr; } .video-card { position:static; } .grid2 { grid-template-columns:1fr; } .gallery { grid-template-columns:1fr; } }
</style>
</head>
<body>
<div class="header">
  <h2>Clean BFA/BFI Dashboard</h2>
  <span class="meta">Data: __DATA_TYPE__</span>
  <span class="meta">Station: __STATION__</span>
  <span class="meta">Samples: __SAMPLES__</span>
  <span class="meta">Features: __FEATURES__</span>
</div>
<div class="dashboard">
  <div class="card video-card">
    <h3>Video</h3>
    <p class="small"><b>Status:</b> __VIDEO_STATUS__</p>
    __VIDEO_HTML__
    <p><input id="timeSlider" type="range" min="__TMIN__" max="__TMAX__" value="__TMIN__" step="0.01" style="width:96%;"></p>
    <p><span id="timeLabel" class="badge">-</span></p>
    <p><span id="windowLabel" class="badge">-</span></p>
    <p><span id="activityLabel" class="badge">-</span></p>
  </div>
  <div>
    <div class="card" id="heatmapCard">
      <h3>BFA/BFI heatmap in a sliding window</h3>
      <p class="small">This heatmap shows the BFA feature matrix over time. X-axis = video time; Y-axis = BFA features, usually subcarrier/group × angle; color = normalized BFA value.</p>
      <div id="heatmap" style="width:100%;height:38vh;"></div>
    </div>
    <div class="grid2">
      <div class="card"><h3>Magnitude and temporal variation</h3><p class="small"><b>RMS</b> summarizes each timestamp into one magnitude value by taking the root mean square across all BFA features. <b>Delta</b> shows how much the BFA pattern changes from one sample to the next.</p><div id="lineplot" style="width:100%;height:28vh;"></div></div>
      <div class="card"><h3>Cumulative samples</h3><p class="small">This plot shows how many valid BFA samples have appeared up to each video time. The fraction line goes from 0 to 1, where 1 means 100% of mapped samples.</p><div id="cumplot" style="width:100%;height:28vh;"></div></div>
    </div>
    <div class="card" style="margin-top:12px;"><h3>Angle/component-wise variation</h3><p class="note small">This plot compares how fast each BFA angle/component changes over time. Values are normalized to 0–1 only to make the curves comparable; they are not physical degrees or signal power.</p><div id="angleplot" style="width:100%;height:30vh;"></div></div>
  </div>
</div>
<div class="tables"><div class="card"><h3>Activity heatmaps</h3><p class="small">Each activity heatmap shows the BFA feature pattern only during samples mapped to that activity group. Use them for visual comparison: stronger color changes or stripe changes suggest that the Wi-Fi feedback pattern changed more during that activity.</p><p class="small"><b>Boxplots:</b> this clean dashboard does not generate boxplots. In older versions, boxplots summarized the distribution of a metric such as RMS per activity, but they were removed because they can hide important time/subcarrier details.</p></div></div>
<div class="gallery">__ACTIVITY_GALLERY__</div>
<div class="tables"><div class="card"><h3>Final tables</h3><h4>Inputs used</h4>__INPUTS_TABLE__<h4>Joined array and 2D matrix summary</h4>__SUMMARY_TABLE__<h4>Example from the first split array</h4>__SPLIT_EXAMPLE__<h4>Example of the 2D matrix used for the heatmap</h4>__MATRIX_EXAMPLE____MAPPING_SECTION__</div></div>
<script>
const payload = __PAYLOAD__;
function clamp(x,a,b){return Math.min(Math.max(x,a),b);}
function left(arr,v){let lo=0,hi=arr.length;while(lo<hi){let m=Math.floor((lo+hi)/2);if(arr[m]<v)lo=m+1;else hi=m;}return lo;}
function right(arr,v){let lo=0,hi=arr.length;while(lo<hi){let m=Math.floor((lo+hi)/2);if(arr[m]<=v)lo=m+1;else hi=m;}return lo;}
function bounds(t){let s=clamp(t-payload.windowSec,payload.tmin,payload.tmax), e=clamp(t,payload.tmin,payload.tmax); if(e<=s)e=Math.min(s+payload.windowSec,payload.tmax); return [s,e];}
function activity(t){for(const a of payload.activities){if(t>=a.start && t<=a.end)return a.group+" / "+a.label;}return "-";}
function sliceZ(i0,i1){let out=[]; for(let r=0;r<payload.z.length;r++)out.push(payload.z[r].slice(i0,i1)); return out;}
function vline(t){return [{type:"line",x0:t,x1:t,y0:0,y1:1,yref:"paper",line:{width:3,color:"black"}}];}
function update(rawT){
  const t=clamp(rawT,payload.tmin,payload.tmax); const b=bounds(t), s=b[0], e=b[1];
  let i0=left(payload.x,s), i1=right(payload.x,e); if(i1<=i0)i1=Math.min(i0+1,payload.x.length);
  const xw=payload.x.slice(i0,i1);
  if(payload.showMainHeatmap){Plotly.restyle("heatmap", {x:[xw], z:[sliceZ(i0,i1)]}, [0]); Plotly.relayout("heatmap", {"xaxis.range":[s,e], "shapes":vline(t)});}
  Plotly.restyle("lineplot", {x:[xw,xw], y:[payload.rms.slice(i0,i1),payload.delta.slice(i0,i1)]}, [0,1]);
  Plotly.relayout("lineplot", {"xaxis.range":[s,e], "shapes":vline(t)}); Plotly.relayout("cumplot", {"shapes":vline(t)});
  if(payload.hasAnglePlot){let xs=[],ys=[];for(let a=0;a<payload.angleNames.length;a++){xs.push(xw);ys.push(payload.angleDelta[a].slice(i0,i1));}Plotly.restyle("angleplot", {x:xs,y:ys}, payload.angleNames.map((_,i)=>i));Plotly.relayout("angleplot", {"xaxis.range":[s,e], "shapes":vline(t)});}
  document.getElementById("timeSlider").value=t; document.getElementById("timeLabel").innerText=t.toFixed(2)+"s"; document.getElementById("windowLabel").innerText="Window: "+s.toFixed(2)+"–"+e.toFixed(2)+"s"; document.getElementById("activityLabel").innerText="Activity: "+activity(rawT);
}
let s0=payload.tmin,e0=Math.min(payload.tmin+payload.windowSec,payload.tmax),i0=left(payload.x,s0),i1=Math.max(right(payload.x,e0),i0+1);
if(payload.showMainHeatmap){Plotly.newPlot("heatmap", [{x:payload.x.slice(i0,i1), y:payload.y, z:sliceZ(i0,i1), type:"heatmap", colorbar:{title:"Intensity"}}], {title:"BFA/BFI heatmap", xaxis:{title:"Video time (s)", range:[s0,e0]}, yaxis:{title:"BFA/BFI features"}, margin:{l:80,r:40,t:55,b:55}, shapes:vline(payload.tmin)});} else {document.getElementById("heatmapCard").style.display="none";}
Plotly.newPlot("lineplot", [{x:payload.x.slice(i0,i1), y:payload.rms.slice(i0,i1), type:"scatter", mode:"lines", name:"RMS"},{x:payload.x.slice(i0,i1), y:payload.delta.slice(i0,i1), type:"scatter", mode:"lines", name:"Delta"}], {title:"Magnitude and Delta", xaxis:{title:"Video time (s)", range:[s0,e0]}, yaxis:{title:"Value"}, margin:{l:70,r:30,t:50,b:55}, shapes:vline(payload.tmin)});
Plotly.newPlot("cumplot", [{x:payload.cumTimes, y:payload.cumCount, type:"scatter", mode:"lines", name:"Cumulative count"},{x:payload.cumTimes, y:payload.cumFraction, type:"scatter", mode:"lines", name:"Cumulative fraction", yaxis:"y2"}], {title:"Cumulative samples", xaxis:{title:"Video time (s)"}, yaxis:{title:"Count"}, yaxis2:{title:"Fraction", overlaying:"y", side:"right", range:[0,1]}, margin:{l:70,r:70,t:50,b:55}, shapes:vline(payload.tmin)});
if(payload.hasAnglePlot){let traces=[];for(let a=0;a<payload.angleNames.length;a++)traces.push({x:payload.x.slice(i0,i1), y:payload.angleDelta[a].slice(i0,i1), type:"scatter", mode:"lines", name:payload.angleNames[a]});Plotly.newPlot("angleplot", traces, {title:"Normalized temporal variation by angle/component", xaxis:{title:"Video time (s)", range:[s0,e0]}, yaxis:{title:"Normalized variation"}, margin:{l:70,r:30,t:50,b:55}, shapes:vline(payload.tmin)});} else {document.getElementById("angleplot").innerHTML="<p>Not applicable for this array.</p>";}
document.getElementById("timeSlider").addEventListener("input",()=>update(parseFloat(document.getElementById("timeSlider").value))); const video=document.getElementById("video"); if(video){video.addEventListener("timeupdate",()=>update(video.currentTime)); video.addEventListener("seeked",()=>update(video.currentTime));} update(payload.tmin);
</script>
</body>
</html>
"""

html_text = html_template
for key, value in {
    "__DATA_TYPE__": PREFERRED_SPLIT_TYPE,
    "__STATION__": str(TARGET_STATION),
    "__SAMPLES__": str(num_samples),
    "__FEATURES__": str(num_features),
    "__VIDEO_STATUS__": video_status,
    "__VIDEO_HTML__": video_html,
    "__TMIN__": f"{payload['tmin']:.4f}",
    "__TMAX__": f"{payload['tmax']:.4f}",
    "__ACTIVITY_GALLERY__": activity_gallery_html,
    "__INPUTS_TABLE__": inputs_html,
    "__SUMMARY_TABLE__": summary_html,
    "__SPLIT_EXAMPLE__": split_example_html,
    "__MATRIX_EXAMPLE__": matrix_example_html,
    "__MAPPING_SECTION__": mapping_section,
    "__PAYLOAD__": json.dumps(payload, ensure_ascii=False),
}.items():
    html_text = html_text.replace(key, value)

OUTPUT_HTML.write_text(html_text, encoding="utf-8")

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUTPUT_HTML, OUTPUT_HTML.name)

print("HTML generated:", OUTPUT_HTML)
print("ZIP backup generated:", OUTPUT_ZIP)
print("HTML size MB:", OUTPUT_HTML.stat().st_size / (1024 * 1024))
display(HTML(f"<p><b>HTML:</b> <code>{OUTPUT_HTML}</code></p><p><b>Temporary output folder:</b> <code>{OUTPUT_DIR}</code></p>"))

HTML generated: /content/bfa_bfi_dashboard_outputs/bfa_bfi_dashboard.html
ZIP backup generated: /content/bfa_bfi_dashboard_outputs/bfa_bfi_dashboard.zip
HTML size MB: 71.26278400421143


## 8. Download the HTML

In [84]:
if IN_COLAB:
    files.download(str(OUTPUT_HTML))
else:
    print("Generated HTML:", OUTPUT_HTML)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>